In [1]:
# p.609~610 | 예제 10.1 | FashionMNIST 다운로드

from itertools import chain
from collections import defaultdict
from torch.utils.data import Subset
from torchvision import datasets


def subset_sampler(dataset, classes, max_len):
    target_idx = defaultdict(list)
    for idx, label in enumerate(dataset.train_labels):
        target_idx[int(label)].append(idx)

    indices = list(
        chain.from_iterable(
            [target_idx[idx][:max_len] for idx in range(len(classes))]
        )
    )
    return Subset(dataset, indices)


train_dataset = datasets.FashionMNIST(
    root="../datasets", download=True, train=True
)
test_dataset = datasets.FashionMNIST(
    root="../datasets", download=True, train=False
)

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

print(classes)
print(class_to_idx)

subset_train_dataset = subset_sampler(
    dataset=train_dataset,
    classes=train_dataset.classes,
    max_len=1000
)

subset_test_dataset = subset_sampler(
    dataset=test_dataset,
    classes=test_dataset.classes,
    max_len=100
)

print(f"Training Data Size : {len(subset_train_dataset)}")
print(f"Testing Data Size : {len(subset_test_dataset)}")
print(train_dataset[0])

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.39MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 140kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.55MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.9MB/s]


['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


/usr/local/lib/python3.13/dist-packages/torchvision/datasets/mnist.py:66: UserWarning: train_labels has been renamed targets
  warnings.warn("train_labels has been renamed targets")


Training Data Size : 10000
Testing Data Size : 1000
(<PIL.Image.Image image mode=L size=28x28 at 0x7A76F15C42D0>, 9)


In [4]:
# p.611 | 예제 10.2 | 이미지 전처리

import torch
from torchvision import transforms
from transformers import AutoImageProcessor


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="google/vit-base-patch16-224-in21k"
)


transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(
            size=(
                image_processor.size["height"],
                image_processor.size["width"]
            )
        ),
        transforms.Lambda(
            lambda x: torch.cat([x, x, x], 0)
        ),
        transforms.Normalize(
            mean=image_processor.image_mean,
            std=image_processor.image_std
        )
    ]
)


print(f"size : {image_processor.size}")
print(f"mean : {image_processor.image_mean}")
print(f"std : {image_processor.image_std}")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

size : SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None, min_pixels=None, max_pixels=None)
mean : (0.5, 0.5, 0.5)
std : (0.5, 0.5, 0.5)


In [6]:
# p.612~613 | 예제 10.3 | ViT 데이터로더 적용

from torch.utils.data import DataLoader


def collator(data, transform):
    images, labels = zip(*data)

    pixel_values = torch.stack(
        [transform(image) for image in images]
    )
    labels = torch.tensor(
        [label for label in labels]
    )

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }


train_dataloader = DataLoader(
    subset_train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda x: collator(x, transform),
    drop_last=True
)

valid_dataloader = DataLoader(
    subset_test_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda x: collator(x, transform),
    drop_last=True
)


batch = next(iter(train_dataloader))

for key, value in batch.items():
    print(f"{key} : {value.shape}")

pixel_values : torch.Size([32, 3, 224, 224])
labels : torch.Size([32])


In [7]:
# p.613~614 | 예제 10.4 | 사전 학습된 ViT 모델

from transformers import ViTForImageClassification


model = ViTForImageClassification.from_pretrained(
    pretrained_model_name_or_path="google/vit-base-patch16-224-in21k",
    num_labels=len(classes),
    id2label={
        idx: label
        for label, idx in class_to_idx.items()
    },
    label2id=class_to_idx,
    ignore_mismatched_sizes=True
)


print(model.classifier)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=10, bias=True)


In [8]:
# p.614~615 | 예제 10.5 | 패치 임베딩 확인

print(model.vit.embeddings)


batch = next(iter(train_dataloader))

print(
    "image shape :",
    batch["pixel_values"].shape
)

print(
    "patch embeddings shape :",
    model.vit.embeddings.patch_embeddings(
        batch["pixel_values"]
    ).shape
)

print(
    "[CLS] + patch embeddings shape :",
    model.vit.embeddings(
        batch["pixel_values"]
    ).shape
)

ViTEmbeddings(
  (patch_embeddings): ViTPatchEmbeddings(
    (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (dropout): Dropout(p=0.0, inplace=False)
)
image shape : torch.Size([32, 3, 224, 224])
patch embeddings shape : torch.Size([32, 196, 768])
[CLS] + patch embeddings shape : torch.Size([32, 197, 768])


In [9]:
# p.615~616 | 예제 10.6 | 하이퍼파라미터 설정

from transformers import TrainingArguments


args = TrainingArguments(
    output_dir="../models/ViT-FashionMNIST",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="logs",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [11]:
# p.617~618 | 예제 10.7 | 매크로 평균 F1 점수

import evaluate
import numpy as np


def compute_metrics(eval_pred):
    metric = evaluate.load("f1")

    predictions, labels = eval_pred
    predictions = np.argmax(
        predictions,
        axis=1
    )

    macro_f1 = metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return macro_f1

ModuleNotFoundError: No module named 'evaluate'

In [12]:
# p.618~621 | 예제 10.8 | ViT 모델 학습

import torch
import evaluate
import numpy as np

from itertools import chain
from collections import defaultdict

from torch.utils.data import Subset
from torchvision import datasets
from torchvision import transforms

from transformers import AutoImageProcessor
from transformers import ViTForImageClassification
from transformers import TrainingArguments, Trainer


def subset_sampler(dataset, classes, max_len):
    target_idx = defaultdict(list)

    for idx, label in enumerate(dataset.train_labels):
        target_idx[int(label)].append(idx)

    indices = list(
        chain.from_iterable(
            [target_idx[idx][:max_len] for idx in range(len(classes))]
        )
    )

    return Subset(dataset, indices)


def model_init(classes, class_to_idx):
    model = ViTForImageClassification.from_pretrained(
        pretrained_model_name_or_path="google/vit-base-patch16-224-in21k",
        num_labels=len(classes),
        id2label={
            idx: label
            for label, idx in class_to_idx.items()
        },
        label2id=class_to_idx
    )

    return model


def collator(data, transform):
    images, labels = zip(*data)

    pixel_values = torch.stack(
        [transform(image) for image in images]
    )

    labels = torch.tensor(
        [label for label in labels]
    )

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }


def compute_metrics(eval_pred):
    metric = evaluate.load("f1")

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return macro_f1


train_dataset = datasets.FashionMNIST(
    root="../datasets",
    download=True,
    train=True
)

test_dataset = datasets.FashionMNIST(
    root="../datasets",
    download=True,
    train=False
)


classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx


subset_train_dataset = subset_sampler(
    dataset=train_dataset,
    classes=train_dataset.classes,
    max_len=1000
)

subset_test_dataset = subset_sampler(
    dataset=test_dataset,
    classes=test_dataset.classes,
    max_len=100
)


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="google/vit-base-patch16-224-in21k"
)


transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(
            size=(
                image_processor.size["height"],
                image_processor.size["width"]
            )
        ),
        transforms.Lambda(
            lambda x: torch.cat([x, x, x], 0)
        ),
        transforms.Normalize(
            mean=image_processor.image_mean,
            std=image_processor.image_std
        )
    ]
)


args = TrainingArguments(
    output_dir="../models/ViT-FashionMNIST",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="logs",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7
)


trainer = Trainer(
    model_init=lambda x: model_init(
        classes,
        class_to_idx
    ),
    args=args,
    train_dataset=subset_train_dataset,
    eval_dataset=subset_test_dataset,
    data_collator=lambda x: collator(
        x,
        transform
    ),
    compute_metrics=compute_metrics,
    tokenizer=image_processor
)

trainer.train()

ModuleNotFoundError: No module named 'evaluate'

In [14]:
# p.622 | 예제 10.9 | ViT 모델 성능 평가

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


outputs = trainer.predict(subset_test_dataset)
print(outputs)


y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)


labels = list(classes)

matrix = confusion_matrix(
    y_true,
    y_pred
)

display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=labels
)

_, ax = plt.subplots(figsize=(10, 10))

display.plot(
    xticks_rotation=45,
    ax=ax
)

plt.show()

NameError: name 'trainer' is not defined

In [15]:
# p.632~633 | 예제 10.10 | 상대적 위치 편향 계산

import torch


window_size = 2

coords_h = torch.arange(window_size)
coords_w = torch.arange(window_size)

coords = torch.stack(
    torch.meshgrid(
        [coords_h, coords_w],
        indexing="ij"
    )
)

coords_flatten = torch.flatten(
    coords,
    1
)

relative_coords = (
    coords_flatten[:, :, None]
    - coords_flatten[:, None, :]
)


print(relative_coords)
print(relative_coords.shape)

tensor([[[ 0,  0, -1, -1],
         [ 0,  0, -1, -1],
         [ 1,  1,  0,  0],
         [ 1,  1,  0,  0]],

        [[ 0, -1,  0, -1],
         [ 1,  0,  1,  0],
         [ 0, -1,  0, -1],
         [ 1,  0,  1,  0]]])
torch.Size([2, 4, 4])


In [16]:
# p.633~634 | 예제 10.11 | X, Y축에 대한 위치 행렬

x_coords = relative_coords[0, :, :]
y_coords = relative_coords[1, :, :]


x_coords += window_size - 1
y_coords += window_size - 1

x_coords *= 2 * window_size - 1


print(
    f"X축에 대한 행렬:\n{x_coords}\n"
)

print(
    f"Y축에 대한 행렬:\n{y_coords}\n"
)


relative_position_index = (
    x_coords + y_coords
)

print(
    f"X, Y축에 대한 위치 행렬:\n{relative_position_index}"
)

X축에 대한 행렬:
tensor([[3, 3, 0, 0],
        [3, 3, 0, 0],
        [6, 6, 3, 3],
        [6, 6, 3, 3]])

Y축에 대한 행렬:
tensor([[1, 0, 1, 0],
        [2, 1, 2, 1],
        [1, 0, 1, 0],
        [2, 1, 2, 1]])

X, Y축에 대한 위치 행렬:
tensor([[4, 3, 1, 0],
        [5, 4, 2, 1],
        [7, 6, 4, 3],
        [8, 7, 5, 4]])


In [17]:
# p.634 | 예제 10.12 | X, Y축에 대한 상대적 위치 좌표 변환

num_heads = 1

relative_position_bias_table = torch.Tensor(
    torch.zeros(
        (2 * window_size - 1)
        * (2 * window_size - 1),
        num_heads
    )
)


relative_position_bias = relative_position_bias_table[
    relative_position_index.view(-1)
]

relative_position_bias = relative_position_bias.view(
    window_size * window_size,
    window_size * window_size,
    -1
)

print(relative_position_bias.shape)

torch.Size([4, 4, 1])


In [18]:
# p.635~636 | 예제 10.13 | 사전 학습된 스윈 트랜스포머 모델

from transformers import SwinForImageClassification


model = SwinForImageClassification.from_pretrained(
    pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224",
    num_labels=len(train_dataset.classes),
    id2label={
        idx: label
        for label, idx in train_dataset.class_to_idx.items()
    },
    label2id=train_dataset.class_to_idx,
    ignore_mismatched_sizes=True
)


for main_name, main_module in model.named_children():
    print(main_name)

    for sub_name, sub_module in main_module.named_children():
        print("└ ", sub_name)

        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └ ", ssub_name)

            for sssub_name, sssub_module in ssub_module.named_children():
                if sssub_name == "projection":
                    print(
                        "│  │  └ ",
                        sssub_name,
                        sssub_module
                    )
                else:
                    print(
                        "│  │  └ ",
                        sssub_name
                    )

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


swin
└  embeddings
│  └  patch_embeddings
│  │  └  projection Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
│  └  norm
│  └  dropout
└  encoder
│  └  layers
│  │  └  0
│  │  └  1
│  │  └  2
│  │  └  3
└  layernorm
└  pooler
classifier


In [19]:
# p.636~637 | 예제 10.14 | 패치 임베딩 모듈

batch = next(iter(train_dataloader))

print(
    "이미지 차원 :",
    batch["pixel_values"].shape
)


patch_emb_output, shape = (
    model.swin.embeddings.patch_embeddings(
        batch["pixel_values"]
    )
)


print(
    "모듈:",
    model.swin.embeddings.patch_embeddings
)

print(
    "패치 임베딩 차원 :",
    patch_emb_output.shape
)

이미지 차원 : torch.Size([32, 3, 224, 224])
모듈: SwinPatchEmbeddings(
  (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
)
패치 임베딩 차원 : torch.Size([32, 3136, 96])


In [20]:
# p.637~638 | 예제 10.15 | 스윈 트랜스포머 블록

for main_name, main_module in (
    model.swin.encoder.layers[0].named_children()
):
    print(main_name)

    for sub_name, sub_module in main_module.named_children():
        print("└ ", sub_name)

        for ssub_name, ssub_module in sub_module.named_children():
            print(
                "│  └ ",
                ssub_name
            )

blocks
└  0
│  └  attention
│  └  layernorm_before
│  └  layernorm_after
│  └  mlp
│  └  dropout
│  └  drop_path
└  1
│  └  attention
│  └  layernorm_before
│  └  layernorm_after
│  └  mlp
│  └  dropout
│  └  drop_path
downsample
└  reduction
└  norm


In [21]:
# p.638 | 예제 10.16 | SwinLayer 구조

print(
    model.swin.encoder.layers[0].blocks[0]
)

SwinLayer(
  (attention): SwinAttention(
    (q_proj): Linear(in_features=96, out_features=96, bias=True)
    (k_proj): Linear(in_features=96, out_features=96, bias=True)
    (v_proj): Linear(in_features=96, out_features=96, bias=True)
    (o_proj): Linear(in_features=96, out_features=96, bias=True)
    (relative_position_bias): SwinRelativePositionBias()
  )
  (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (layernorm_after): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (mlp): SwinMLP(
    (activation_fn): GELUActivation()
    (fc1): Linear(in_features=96, out_features=384, bias=True)
    (fc2): Linear(in_features=384, out_features=96, bias=True)
  )
  (dropout): Dropout(p=0.0, inplace=False)
  (drop_path): Identity()
)


In [27]:
# p.640 | 예제 10.17 | W-MSA, SW-MSA 모듈

print(
    "패치 임베딩 차원 :",
    patch_emb_output.shape
)


W_MSA = model.swin.encoder.layers[0].blocks[0]
SW_MSA = model.swin.encoder.layers[0].blocks[1]


W_MSA_output = W_MSA(
    patch_emb_output,
    W_MSA.input_resolution
)[0]

SW_MSA_output = SW_MSA(
    W_MSA_output,
    SW_MSA.input_resolution
)[0]


print(
    "W-MSA 결과 차원 :",
    W_MSA_output.shape
)

print(
    "SW-MSA 결과 차원 :",
    SW_MSA_output.shape
)

패치 임베딩 차원 : torch.Size([32, 3136, 96])
W-MSA 결과 차원 : torch.Size([32, 3136, 96])
SW-MSA 결과 차원 : torch.Size([32, 3136, 96])


In [28]:
# p.640 | 예제 10.18 | 패치 병합

patch_merge = model.swin.encoder.layers[0].downsample

print(
    "patch_merge 모듈 :",
    patch_merge
)


output = patch_merge(
    SW_MSA_output,
    patch_merge.input_resolution
)

print(
    "patch_merge 결과 차원 :",
    output.shape
)

patch_merge 모듈 : SwinPatchMerging(
  (reduction): Linear(in_features=384, out_features=192, bias=False)
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
)


AttributeError: 'SwinPatchMerging' object has no attribute 'input_resolution'

In [29]:
# p.641~642 | 예제 10.19 | 스윈 트랜스포머 모델 학습

from transformers import SwinForImageClassification


...


def model_init(classes, class_to_idx):
    model = SwinForImageClassification.from_pretrained(
        pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224",
        num_labels=len(classes),
        id2label={
            idx: label
            for label, idx in class_to_idx.items()
        },
        label2id=class_to_idx,
        ignore_mismatched_sizes=True
    )

    return model


...


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224"
)


...


args = TrainingArguments(
    output_dir="../models/Swin-FashionMNIST",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="logs",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7
)


...

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [30]:
# p.649~650 | 예제 10.20 | CvT 모델 이미지 데이터 전처리

from transformers import AutoImageProcessor


...


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="microsoft/cvt-21"
)


transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(
            size=(
                image_processor.size["shortest_edge"],
                image_processor.size["shortest_edge"]
            )
        ),
        transforms.Lambda(
            lambda x: torch.cat([x, x, x], 0)
        ),
        transforms.Normalize(
            mean=image_processor.image_mean,
            std=image_processor.image_std
        )
    ]
)


...

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Ellipsis

In [31]:
# p.650~651 | 예제 10.21 | 사전 학습된 CvT 모델

from transformers import CvtForImageClassification


model = CvtForImageClassification.from_pretrained(
    pretrained_model_name_or_path="microsoft/cvt-21",
    num_labels=len(train_dataset.classes),
    id2label={
        idx: label
        for label, idx in train_dataset.class_to_idx.items()
    },
    label2id=train_dataset.class_to_idx,
    ignore_mismatched_sizes=True
)


for main_name, main_module in model.named_children():
    print(main_name)

    for sub_name, sub_module in main_module.named_children():
        print("└ ", sub_name)

        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └ ", ssub_name)

            for sssub_name, sssub_module in ssub_module.named_children():
                print(
                    "│  │  └ ",
                    sssub_name
                )

config.json:   0%|          | 0.00/70.3k [00:00<?, ?B/s]

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  127MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

[transformers] CvtForImageClassification LOAD REPORT from: microsoft/cvt-21
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 384]) vs model:torch.Size([10, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


cvt
└  encoder
│  └  stages
│  │  └  0
│  │  └  1
│  │  └  2
layernorm
classifier


In [32]:
# p.651 | 예제 10.22 | CvT 모델의 스테이지 구조

stages = model.cvt.encoder.stages

print(stages[0])

CvtStage(
  (embedding): CvtEmbeddings(
    (convolution_embeddings): CvtConvEmbeddings(
      (projection): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(2, 2))
      (normalization): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (layers): Sequential(
    (0): CvtLayer(
      (attention): CvtAttention(
        (attention): CvtSelfAttention(
          (convolution_projection_query): CvtSelfAttentionProjection(
            (convolution_projection): CvtSelfAttentionConvProjection(
              (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
              (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (linear_projection): CvtSelfAttentionLinearProjection()
          )
          (convolution_projection_key): CvtSelfAttentionProjection(
            (convolution_projection): CvtSelfAtte

In [33]:
# p.653~654 | 예제 10.23 | 셀프 어텐션 적용

batch = next(iter(train_dataloader))

print(
    "이미지 차원 :",
    batch["pixel_values"].shape
)


patch_emb_output = stages[0].embedding(
    batch["pixel_values"]
)

print(
    "패치 임베딩 차원 :",
    patch_emb_output.shape
)


batch_size, num_channels, height, width = (
    patch_emb_output.shape
)

hidden_state = patch_emb_output.view(
    batch_size,
    num_channels,
    height * width
).permute(0, 2, 1)


print(
    "셀프 어텐션 입력 차원 :",
    hidden_state.shape
)


attention_output = (
    stages[0]
    .layers[0]
    .attention
    .attention(
        hidden_state,
        height,
        width
    )
)


print(
    "셀프 어텐션 출력 차원 :",
    attention_output.shape
)

이미지 차원 : torch.Size([32, 3, 224, 224])
패치 임베딩 차원 : torch.Size([32, 64, 56, 56])
셀프 어텐션 입력 차원 : torch.Size([32, 3136, 64])
셀프 어텐션 출력 차원 : torch.Size([32, 3136, 64])


In [ ]:
# p.654~656 | 예제 10.24 | CvT 모델 학습

from transformers import CvtForImageClassification


...


def model_init(classes, class_to_idx):
    model = CvtForImageClassification.from_pretrained(
        pretrained_model_name_or_path="microsoft/cvt-21",
        num_labels=len(classes),
        id2label={
            idx: label
            for label, idx in class_to_idx.items()
        },
        label2id=class_to_idx,
        ignore_mismatched_sizes=True
    )

    return model


...


image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path="microsoft/cvt-21"
)


transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(
            size=(
                image_processor.size["shortest_edge"],
                image_processor.size["shortest_edge"]
            )
        ),
        transforms.Lambda(
            lambda x: torch.cat([x, x, x], 0)
        ),
        transforms.Normalize(
            mean=image_processor.image_mean,
            std=image_processor.image_std
        )
    ]
)


...


args = TrainingArguments(
    output_dir="../models/CvT-FashionMNIST",
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="logs",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7
)


...

